Cell 1 - Tải các gói package, thư viện cần thiết

In [ ]:
# Cai dat RF-DETR va cac thu vien can thiet
!pip install -q "rfdetr[train,loggers]" supervision pycocotools opencv-python-headless

import torch
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("So GPU nhin thay:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")


In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

Cell 2 - Load Dataset

In [ ]:
import os
import kagglehub

# Download dataset
path = kagglehub.dataset_download("anhxunv/brain-turmor-segment-datasets")
print("Path to dataset files:", path)

# Scan cấu trúc thư mục
print("\n📁 Cấu trúc thư mục:")
for root, dirs, files in os.walk(path):
    level = root.replace(path, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    if level < 3:  # chỉ in 3 level đầu
        subindent = ' ' * 2 * (level + 1)
        for f in files[:5]:  # in tối đa 5 file
            print(f'{subindent}{f}')
        if len(files) > 5:
            print(f'{subindent}... ({len(files)} files total)')

Cell 3 - Kham pha cau truc dataset & xac dinh dinh dang (COCO JSON / YOLO / anh+mask roi)

In [ ]:
def inspect_dataset(root):
    coco_found, yolo_found = [], []
    ext_count = {}
    sample_files = {}
    for dirpath, dirnames, filenames in os.walk(root):
        for fn in filenames:
            ext = os.path.splitext(fn)[1].lower()
            ext_count[ext] = ext_count.get(ext, 0) + 1
            sample_files.setdefault(ext, []).append(os.path.join(dirpath, fn))
            if fn == "_annotations.coco.json":
                coco_found.append(dirpath)
            if fn in ("data.yaml", "data.yml"):
                yolo_found.append(dirpath)

    print("\U0001F4E6 Thong ke file theo dinh dang:")
    for ext, cnt in sorted(ext_count.items(), key=lambda x: -x[1]):
        print(f"  {ext or '(no ext)'}: {cnt} file")

    print("\n\U0001F50D Phat hien format:")
    if coco_found:
        print("  OK - COCO JSON tai:", coco_found)
    if yolo_found:
        print("  OK - YOLO data.yaml tai:", yolo_found)
    if not coco_found and not yolo_found:
        print("  CANH BAO: khong tim thay _annotations.coco.json hoac data.yaml")
        print("  -> Dataset co the dang o dang anh + mask roi, can convert o Cell 4.")

    print("\n\U0001F4C1 Mot vai file mau cho moi extension:")
    for ext, files in sample_files.items():
        print(f"  {ext}: {files[:3]}")
    return ext_count, coco_found, yolo_found

DATASET_ROOT = path  # bien `path` co tu Cell 2 (kagglehub.dataset_download)
ext_count, coco_found, yolo_found = inspect_dataset(DATASET_ROOT)


Cell 4 - Chuan bi dataset dung format cho RF-DETR (Instance Segmentation)

**Quan trong:** RF-DETR can dataset o dang COCO JSON hoac YOLO, voi cau truc `train/`, `valid/`, `test/` moi thu muc co `_annotations.coco.json` (hoac `data.yaml` cho YOLO).

- Neu Cell 3 da tim thay COCO/YOLO co san -> cell duoi se tu dung luon, khong can sua gi.
- Neu dataset cua ban la **anh + mask roi rac** (pho bien voi cac dataset segmentation y te tren Kaggle) -> sua 3 bien `IMAGES_DIR`, `MASKS_DIR`, `CLASS_NAME` cho dung voi output cua Cell 3, roi chay lai cell de tu dong convert sang COCO polygon format.

In [ ]:
import json
import shutil

import cv2
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

OUTPUT_DATASET_DIR = "/kaggle/working/dataset_coco"  # dataset sau khi convert, RF-DETR train tu day
CLASS_NAME = "tumor"  # dataset nay la binary segmentation (mask chi co 2 gia tri 0/255)

# ============================================================
# TRUONG HOP A - dataset da co _annotations.coco.json hoac data.yaml co san
# ============================================================
if coco_found:
    DATASET_DIR = os.path.dirname(coco_found[0])
    print(f"Dung truc tiep dataset COCO co san tai: {DATASET_DIR}")
elif yolo_found:
    DATASET_DIR = yolo_found[0]
    print(f"Dung truc tiep dataset YOLO co san tai: {DATASET_DIR}")
else:
    # === CACHE: tranh convert lai tu dau neu da chay cell nay roi ===
    # (qua trinh doc tung mask + tim contour + ghi anh ra dia kha ton thoi gian,
    #  nen neu OUTPUT_DATASET_DIR da co du 3 split + file _annotations.coco.json
    #  thi bo qua, dung lai luon - giup chay lai notebook nhanh hon rat nhieu)
    _splits_expected = ("train", "valid", "test")
    _cache_ok = all(
        os.path.exists(os.path.join(OUTPUT_DATASET_DIR, _s, "_annotations.coco.json"))
        for _s in _splits_expected
    )
    if _cache_ok:
        DATASET_DIR = OUTPUT_DATASET_DIR
        print(f"Da tim thay dataset COCO da convert tu lan chay truoc tai: {DATASET_DIR}")
        print("-> Bo qua buoc convert mask de tiet kiem thoi gian.")
        print("   (Neu muon convert lai tu dau - vi du doi CLASS_NAME hoac sua mask - "
              "hay xoa thu muc nay roi chay lai cell nay.)")
    else:
        # ========================================================
        # TRUONG HOP B - anh + mask roi (cau truc thuc te cua
        # anhxunv/brain-turmor-segment-datasets, da xac nhan tu notebook
        # Attention U-Net tham khao da chay thuc te tren Kaggle):
        # <DATASET_ROOT>/datasets/image/ va <DATASET_ROOT>/datasets/mask/,
        # TEN FILE GIONG NHAU giua 2 thu muc, mask la binary (0/255).
        # ========================================================
        IMAGES_DIR = os.path.join(DATASET_ROOT, "datasets", "image")
        MASKS_DIR = os.path.join(DATASET_ROOT, "datasets", "mask")
        IMAGE_EXTENSIONS = (".png", ".jpg", ".jpeg")

        def get_sorted_files(folder):
            return [
                os.path.join(folder, f)
                for f in sorted(os.listdir(folder))
                if f.lower().endswith(IMAGE_EXTENSIONS)
            ]

        image_paths = get_sorted_files(IMAGES_DIR)
        mask_paths = get_sorted_files(MASKS_DIR)

        assert len(image_paths) == len(mask_paths) and len(image_paths) > 0, (
            "So anh va so mask khong khop - kiem tra lai IMAGES_DIR / MASKS_DIR."
        )
        for img_f, mask_f in zip(image_paths[:5], mask_paths[:5]):
            assert os.path.basename(img_f) == os.path.basename(mask_f), (
                f"Ten file khong khop: {os.path.basename(img_f)} vs {os.path.basename(mask_f)}"
            )
        print(f"Tim thay {len(image_paths)} cap anh-mask (ten file giong nhau giua 2 thu muc).")

        # --- Kiem tra nhanh 3 cap anh-mask dau, xem mask co AN KHOP voi vung benh khong ---
        print("\nKiem tra nhanh 3 cap anh-mask dau:")
        for _img_path, _mask_path in list(zip(image_paths, mask_paths))[:3]:
            _img = cv2.cvtColor(cv2.imread(_img_path), cv2.COLOR_BGR2RGB)
            _mask = cv2.imread(_mask_path, cv2.IMREAD_GRAYSCALE)
            _overlay = _img.copy()
            _overlay[_mask > 127] = [255, 0, 0]
            _fig, _axes = plt.subplots(1, 3, figsize=(12, 4))
            _axes[0].imshow(_img); _axes[0].set_title(os.path.basename(_img_path)); _axes[0].axis("off")
            _axes[1].imshow(_mask, cmap="gray"); _axes[1].set_title(os.path.basename(_mask_path)); _axes[1].axis("off")
            _axes[2].imshow(_overlay); _axes[2].set_title("Overlay (do = mask)"); _axes[2].axis("off")
            plt.show()

        # --- Split train/val/test 80/10/10, GIONG HET notebook Attention U-Net ---
        # (random_state=42 -> cung mot bo test giua RF-DETR va U-Net, so sanh cong bang)
        img_train, img_temp, mask_train, mask_temp = train_test_split(
            image_paths, mask_paths, test_size=0.2, random_state=42
        )
        img_val, img_test, mask_val, mask_test = train_test_split(
            img_temp, mask_temp, test_size=0.5, random_state=42
        )
        splits = {
            "train": list(zip(img_train, mask_train)),
            "valid": list(zip(img_val, mask_val)),
            "test": list(zip(img_test, mask_test)),
        }
        print("\nTrain/Val/Test split (80/10/10, random_state=42 - dong bo voi notebook U-Net):")
        for _name, _split_pairs in splits.items():
            print(f"  {_name}: {len(_split_pairs)} samples")

        def mask_to_coco_annotations(mask_path, image_id, ann_id_start):
            mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
            _, binary = cv2.threshold(mask, 127, 255, cv2.THRESH_BINARY)
            contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            anns = []
            ann_id = ann_id_start
            for cnt in contours:
                if cv2.contourArea(cnt) < 10:  # bo vung nhieu qua nho
                    continue
                x, y, w, h = cv2.boundingRect(cnt)
                segmentation = cnt.flatten().tolist()
                if len(segmentation) < 6:
                    continue
                anns.append({
                    "id": ann_id,
                    "image_id": image_id,
                    "category_id": 1,
                    "bbox": [x, y, w, h],
                    "area": float(cv2.contourArea(cnt)),
                    "iscrowd": 0,
                    "segmentation": [segmentation],
                })
                ann_id += 1
            return anns, ann_id

        for split_name, split_pairs in splits.items():
            split_dir = os.path.join(OUTPUT_DATASET_DIR, split_name)
            os.makedirs(split_dir, exist_ok=True)
            images_meta, annotations = [], []
            ann_id = 1
            for image_id, (img_path, mask_path) in enumerate(split_pairs, start=1):
                img = cv2.imread(img_path)
                h, w = img.shape[:2]
                fname = f"{image_id:06d}.jpg"
                cv2.imwrite(os.path.join(split_dir, fname), img)
                images_meta.append({"id": image_id, "file_name": fname, "width": w, "height": h})
                anns, ann_id = mask_to_coco_annotations(mask_path, image_id, ann_id)
                annotations.extend(anns)

            coco_dict = {
                "info": {"description": "anhxunv/brain-turmor-segment-datasets, converted to COCO"},
                "licenses": [],
                "images": images_meta,
                "categories": [{"id": 1, "name": CLASS_NAME, "supercategory": "none"}],
                "annotations": annotations,
            }
            with open(os.path.join(split_dir, "_annotations.coco.json"), "w") as f:
                json.dump(coco_dict, f)
            print(f"  {split_name}: {len(images_meta)} anh, {len(annotations)} annotation")

        DATASET_DIR = OUTPUT_DATASET_DIR
        print(f"\nDa convert dataset sang COCO format tai: {DATASET_DIR}")

    print("\nDATASET_DIR cuoi cung dung de train:", DATASET_DIR)


Cell 5 - Cau hinh model RF-DETR (Segmentation) & cac duong dan output

Mac dinh dung **RFDETRSegMedium** (33.7M params, do phan giai 432x432) - can bang tot giua toc do va do chinh xac tren 2 GPU T4. Doi sang `RFDETRSegNano/Small/Large/XLarge/2XLarge` neu can nhanh hon hoac chinh xac hon (XL/2XL can `pip install rfdetr[plus]` va co license PML 1.0 rieng).

In [ ]:
from rfdetr import RFDETRSegSmall  # THAY DOI: Small thay vi Medium (giam tu 33.7M xuong 17.9M params)

OUTPUT_DIR = "/kaggle/working/rfdetr_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

EPOCHS = 60
BATCH_SIZE = 8  # TANG len 8 vi Small model nhe hon, T4 16GB du VRAM
GRAD_ACCUM_STEPS = 1  # GIAM xuong 1 (effective batch = 8 x 1 x 1 = 8, du cho binary segmentation)
CHECKPOINT_INTERVAL = 5

NUM_WORKERS = min(4, os.cpu_count() or 2)
PIN_MEMORY = True
PERSISTENT_WORKERS = True
PREFETCH_FACTOR = 4
PROGRESS_BAR = "tqdm"
EVAL_INTERVAL = 3  # validate moi 3 epoch

MULTI_SCALE = False  # tat multi-scale augmentation
EXPANDED_SCALES = False

resume_path = None
last_ckpt = os.path.join(OUTPUT_DIR, "last.ckpt")
if os.path.exists(last_ckpt):
    resume_path = last_ckpt
    print(f"Tim thay checkpoint truoc do, se RESUME tu: {resume_path}")
else:
    print("Khong co checkpoint truoc do - train tu dau (tu pretrained COCO weights).")

best_weights_path = os.path.join(OUTPUT_DIR, "best_weights.pth")
last_weights_path = os.path.join(OUTPUT_DIR, "last_weights.ckpt")

print(f"NUM_WORKERS = {NUM_WORKERS} (CPU co san: {os.cpu_count()})")
print(f"MODEL: RFDETRSegSmall (17.9M params, resolution 320x320)")
print(f"TOI UU: EVAL_INTERVAL=3, MULTI_SCALE=False, BATCH_SIZE=8, GRAD_ACCUM=1")
print(f"=> Du kien train nhanh hon ~70% (tu 18h xuong ~5-6h)")

model = RFDETRSegSmall()  # THAY DOI o day nua


**Cell 5b - "Nguoi gac" tu dong luu best_weights / last_weights moi epoch + in tien trinh**

RF-DETR (ban moi, dung PyTorch Lightning) da tu dong:
- Luu `last.ckpt` lai moi epoch (du de resume).
- Luu `checkpoint_best_regular.pth` / `checkpoint_best_ema.pth` moi khi co mAP cao hon (chi chua weights, khong co optimizer state - dung de predict duoc ngay).
- In bang chi so (mAP, F1, AP50...) ra ngay sau moi epoch validation.

**Van de:** file `best_weights.pth` chinh thuc (`checkpoint_best_total.pth`) chi duoc tao **sau khi train() chay xong toan bo** (ham `on_fit_end`). Neu Kaggle bi ngat session (het gio, mat ket noi, OOM...) giua luc train thi file nay **se khong bao gio duoc tao**, dau model thuc te da train rat tot roi.

Cell duoi tao 1 luong (thread) chay song song voi `model.train()`: cu vai chuc giay se tu dong copy ban checkpoint tot nhat/gan nhat hien co thanh `best_weights.pth` / `last_weights.ckpt`, va in ra 1 dong tom tat moi khi co epoch moi xong - du train co bi ngat dot ngot giua luc chay thi 2 file nay van luon la ban moi nhat co the co.

In [ ]:
import csv
import shutil
import threading
import time
from datetime import datetime


class CheckpointWatcher:
    """Chay song song (1 thread rieng) voi model.train().

    Cu moi `interval` giay se:
      1) Dong bo best_weights.pth = ban checkpoint co mAP cao nhat HIEN TAI (regular hoac EMA, cai nao moi hon).
      2) Dong bo last_weights.ckpt = ban sao cua last.ckpt (checkpoint epoch gan nhat, dung de resume).
      3) Doc metrics.csv, neu co epoch moi thi in ra 1 dong tom tat chi so (mAP, F1...) + thoi gian da train.

    Vi chi doc/ghi file tren dia (khong dong vao noi bo PyTorch Lightning), watcher nay
    hoat dong doc lap, an toan ngay ca khi qua trinh train bi ngat dot ngot giua epoch.
    """

    def __init__(self, output_dir: str, total_epochs: int, interval: int = 20):
        self.output_dir = output_dir
        self.total_epochs = total_epochs
        self.interval = interval
        self._stop_event = threading.Event()
        self._thread = None
        self._last_best_mtime = 0.0
        self._last_ckpt_mtime = 0.0
        self._last_metrics_rows = 0
        self._start_time = time.time()

    def _sync_best_weights(self):
        candidates = [
            os.path.join(self.output_dir, name)
            for name in ("checkpoint_best_total.pth", "checkpoint_best_ema.pth", "checkpoint_best_regular.pth")
            if os.path.exists(os.path.join(self.output_dir, name))
        ]
        if not candidates:
            return
        newest = max(candidates, key=os.path.getmtime)
        mtime = os.path.getmtime(newest)
        if mtime > self._last_best_mtime:
            shutil.copy2(newest, os.path.join(self.output_dir, "best_weights.pth"))
            self._last_best_mtime = mtime

    def _sync_last_weights(self):
        last_ckpt = os.path.join(self.output_dir, "last.ckpt")
        if os.path.exists(last_ckpt):
            mtime = os.path.getmtime(last_ckpt)
            if mtime > self._last_ckpt_mtime:
                shutil.copy2(last_ckpt, os.path.join(self.output_dir, "last_weights.ckpt"))
                self._last_ckpt_mtime = mtime

    def _print_new_epoch_progress(self):
        metrics_path = os.path.join(self.output_dir, "metrics.csv")
        if not os.path.exists(metrics_path):
            return
        try:
            with open(metrics_path, newline="") as f:
                rows = list(csv.DictReader(f))
        except Exception:
            return
        if len(rows) <= self._last_metrics_rows:
            return
        self._last_metrics_rows = len(rows)
        last_row = rows[-1]

        def fmt(key):
            val = last_row.get(key, "")
            try:
                return f"{float(val):.4f}"
            except (TypeError, ValueError):
                return "-"

        epoch = last_row.get("epoch", "?")
        elapsed_min = (time.time() - self._start_time) / 60
        print(
            f"[{datetime.now().strftime('%H:%M:%S')}] "
            f"epoch {epoch}/{self.total_epochs} | da train {elapsed_min:.1f} phut | "
            f"train/loss={fmt('train/loss')} | "
            f"val/mAP_50_95={fmt('val/mAP_50_95')} | "
            f"val/segm_mAP_50_95={fmt('val/segm_mAP_50_95')} | "
            f"val/F1={fmt('val/F1')} "
            f"| best_weights.pth & last_weights.ckpt da duoc cap nhat"
        )

    def _run_loop(self):
        while not self._stop_event.is_set():
            try:
                self._sync_best_weights()
                self._sync_last_weights()
                self._print_new_epoch_progress()
            except Exception as exc:  # khong de watcher lam crash qua trinh train
                print(f"[CheckpointWatcher] loi tam thoi (bo qua, se thu lai sau {self.interval}s): {exc}")
            self._stop_event.wait(self.interval)

    def start(self):
        self._thread = threading.Thread(target=self._run_loop, daemon=True)
        self._thread.start()
        print(f"[CheckpointWatcher] da bat dau - kiem tra checkpoint moi {self.interval}s.")
        return self

    def stop(self):
        self._stop_event.set()
        if self._thread is not None:
            self._thread.join(timeout=10)
        # dong bo lan cuoi de chac chan khong bi sot epoch cuoi
        self._sync_best_weights()
        self._sync_last_weights()
        self._print_new_epoch_progress()
        print("[CheckpointWatcher] da dung - best_weights.pth / last_weights.ckpt la ban moi nhat hien co.")


**Cell 6 - Train voi 2 GPU T4 (DDP ngay trong notebook)**

**Truoc khi chay:** vao Notebook Settings (panel ben phai) -> Accelerator -> chon **GPU T4 x2**, va bat **Internet** (can internet de tai pretrained weights tu Roboflow). RF-DETR co san `strategy="ddp_notebook"` duoc team Roboflow thiet ke de chay multi-GPU an toan truc tiep trong Jupyter/Kaggle, khong can `torchrun` hay file script rieng.

**Trong khi cell nay chay:** ban se thay 1 thanh tien trinh (tqdm) cho tung epoch/batch, kem 1 bang chi so (mAP, F1, AP50...) ma RF-DETR tu in ra sau moi epoch validation, va xen giua la cac dong tom tat tu "nguoi gac" (Cell 5b) bao best_weights.pth / last_weights.ckpt vua duoc cap nhat.

**Cac huong tang toc do (neu van thay qua lau), co the thu them:**
- Doi sang model nho hon: `RFDETRSegSmall` hoac `RFDETRSegNano` (it tham so, do phan giai thap hon -> nhanh hon ro ret, doi lai mAP co the giam mot it).
- Giam do phan giai bang `resolution=...` trong `model.train()` (phai la boi so cua patch_size*num_windows cua model).
- Tat augmentation da ti le: dat `MULTI_SCALE = False` va `EXPANDED_SCALES = False` o Cell 5 - giam dang ke compute moi step nhung co the anh huong nhe den do chinh xac cuoi cung.
- Tang `EVAL_INTERVAL` o Cell 5 (vd. =2) neu chap nhan chi xem chi so mAP moi 2 epoch thay vi moi epoch - buoc validation/segm-mAP kha ton thoi gian.
- So sanh `devices=1` (1 GPU) voi `devices=2`: tren 2x T4 khong co NVLink, chi phi dong bo DDP (`find_unused_parameters=True`) doi khi khien 2 GPU khong nhanh hon 1 GPU bao nhieu, tuy dataset. Nen do thu thoi gian 1 epoch o ca 2 cau hinh truoc khi quyet dinh.

In [ ]:
# Khoi dong "nguoi gac" - chay song song, tu dong cap nhat best_weights.pth / last_weights.ckpt
# + in tien trinh moi epoch (xem Cell 5b ben tren). interval=20s la du nhanh ma khong lam log qua nhieu.
watcher = CheckpointWatcher(OUTPUT_DIR, total_epochs=EPOCHS, interval=20).start()

try:
    model.train(
        dataset_dir=DATASET_DIR,
        output_dir=OUTPUT_DIR,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        grad_accum_steps=GRAD_ACCUM_STEPS,
        lr=1e-4,
        devices=1,  # **TOI UU: chi dung 1 GPU vi model Small + DDP overhead cao hon loi ich tren 2xT4 khong co NVLink**
        strategy="auto",  # auto cho 1 GPU (khong can ddp_notebook)
        checkpoint_interval=CHECKPOINT_INTERVAL,
        use_ema=True,
        early_stopping=True,
        early_stopping_patience=10,
        resume=resume_path,
        tensorboard=True,
        run_test=True,
        # --- cac tham so toi uu toc do / hien thi ---
        progress_bar=PROGRESS_BAR,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        persistent_workers=PERSISTENT_WORKERS,
        prefetch_factor=PREFETCH_FACTOR,
        eval_interval=EVAL_INTERVAL,
        multi_scale=MULTI_SCALE,
        expanded_scales=EXPANDED_SCALES,
    )
finally:
    watcher.stop()


Cell 7 - Luu file `best_weights` va `last_checkpoint_epochXX`

- **best_weights.pth**: chi chua trong so model (khong co optimizer state) - dung de tai tao kien truc + trong so phuc vu predict/inference. Lay tu `checkpoint_best_total.pth` (RF-DETR tu chon ban EMA hoac raw nao co mAP validation cao hon).
- **last_checkpoint_epochXX.ckpt**: checkpoint day du (model + optimizer + scheduler) tai epoch gan nhat - dung de RESUME train tiep neu notebook bi ngat giua luc chay (load file nay vao tham so `resume=`).

In [ ]:
import glob

# Watcher (Cell 5b) da lien tuc cap nhat best_weights.pth va last_weights.ckpt
# trong SUOT qua trinh train (khong phai chi luc ket thuc) - cell nay chi lam
# 1 buoc dong bo CUOI CUNG de bao dam co ban moi nhat, ke ca khi RF-DETR vua
# tao xong checkpoint_best_total.pth (chinh thuc, chon giua regular vs EMA) sau khi train() ket thuc binh thuong.

best_total_path = os.path.join(OUTPUT_DIR, "checkpoint_best_total.pth")
if os.path.exists(best_total_path):
    shutil.copy2(best_total_path, best_weights_path)
    print(f"Da cap nhat best weights (chinh thuc, sau khi train xong) tai: {best_weights_path}")
elif os.path.exists(best_weights_path):
    print(f"Dung ban best_weights.pth ma watcher da luu trong luc train (chua co checkpoint_best_total.pth): {best_weights_path}")
else:
    print("CANH BAO: chua thay best_weights.pth nao - co the chua chay validation lan nao.")

last_ckpt_path = os.path.join(OUTPUT_DIR, "last.ckpt")
if os.path.exists(last_ckpt_path):
    shutil.copy2(last_ckpt_path, last_weights_path)
    ckpt = torch.load(last_ckpt_path, map_location="cpu", weights_only=False)
    last_epoch = ckpt.get("epoch", None)
    suffix = f"{last_epoch:02d}" if isinstance(last_epoch, int) else "unknown"
    last_checkpoint_named = os.path.join(OUTPUT_DIR, f"last_checkpoint_epoch{suffix}.ckpt")
    shutil.copy2(last_ckpt_path, last_checkpoint_named)
    print(f"Da cap nhat last weights tai: {last_weights_path}")
    print(f"Da luu them ban co ten epoch: {last_checkpoint_named}")
else:
    print("CANH BAO: chua thay last.ckpt.")

print("\nToan bo checkpoint trong output_dir:")
for f in sorted(glob.glob(os.path.join(OUTPUT_DIR, "*"))):
    if os.path.isfile(f):
        size_mb = os.path.getsize(f) / 1e6
        print(f"  {os.path.basename(f):40s} {size_mb:8.1f} MB")


Cell 8 - Cac chi so danh gia (mAP, AP50, AP75, AR, F1...)

RF-DETR dung **torchmetrics** + COCO-style evaluation, log vao `metrics.csv` trong output_dir moi epoch. Cac chi so chinh:

- `val/mAP_50_95` - box mAP trung binh tren IoU 0.5:0.95 (chi so chinh, dung de chon best checkpoint va early stopping)
- `val/mAP_50`, `val/mAP_75` - box mAP tai IoU 0.5 va 0.75
- `val/mAR` - mean Average Recall
- `val/segm_mAP_50_95`, `val/segm_mAP_50` - mAP tren MASK (vi day la model segmentation) thay vi chi bounding box
- `val/F1`, `val/precision`, `val/recall` - tu confidence-threshold sweep
- `val/AP/<ten_class>` - AP rieng cho moi class
- Sau khi train xong, cac chi so tren duoc tinh **1 lan tren test set** voi prefix `test/`

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

metrics_path = os.path.join(OUTPUT_DIR, "metrics.csv")
df = pd.read_csv(metrics_path)

metric_cols = [c for c in df.columns if c.startswith("val/") or c.startswith("test/")]
print("Cac cot chi so co trong log:")
print(metric_cols)

# ------------------------------------------------------------------
# 1) LEARNING CURVE - train/loss vs val/loss theo epoch
# ------------------------------------------------------------------
loss_cols = [c for c in ["epoch", "train/loss", "val/loss"] if c in df.columns]
if len(loss_cols) >= 2:
    # metrics.csv co the ghi train/loss va val/loss tren 2 dong khac nhau cung epoch
    # -> gop lai thanh 1 dong/epoch truoc khi ve
    loss_df = df[loss_cols].groupby("epoch", as_index=False).mean()
    plt.figure(figsize=(8, 4))
    if "train/loss" in loss_df.columns:
        plt.plot(loss_df["epoch"], loss_df["train/loss"], marker="o", label="train/loss")
    if "val/loss" in loss_df.columns:
        plt.plot(loss_df["epoch"], loss_df["val/loss"], marker="o", label="val/loss")
    plt.xlabel("epoch")
    plt.ylabel("loss")
    plt.title("Learning curve")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()
else:
    print("Chua thay train/loss hoac val/loss trong metrics.csv (train() chua chay xong?).")

# ------------------------------------------------------------------
# 2) Bang chi so validation theo epoch (mAP, AP50, AP75, AR, F1...)
# ------------------------------------------------------------------
key_cols = [c for c in [
    "epoch", "val/mAP_50_95", "val/mAP_50", "val/mAP_75", "val/mAR",
    "val/segm_mAP_50_95", "val/segm_mAP_50",
    "val/F1", "val/precision", "val/recall",
] if c in df.columns]
display(df[key_cols].dropna(how="all", subset=key_cols[1:]))

# Bieu do mAP qua cac epoch
plot_df = df.dropna(subset=["val/mAP_50_95"]) if "val/mAP_50_95" in df.columns else None
if plot_df is not None and len(plot_df) > 0:
    plt.figure(figsize=(8, 4))
    plt.plot(plot_df["epoch"], plot_df["val/mAP_50_95"], marker="o", label="val/mAP_50_95")
    if "val/segm_mAP_50_95" in plot_df.columns:
        plt.plot(plot_df["epoch"], plot_df["val/segm_mAP_50_95"], marker="o", label="val/segm_mAP_50_95")
    plt.xlabel("epoch")
    plt.ylabel("mAP")
    plt.title("mAP theo epoch (validation)")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()

# ------------------------------------------------------------------
# 3) KET QUA TREN TEST SET - chi chay 1 lan duy nhat sau khi train() xong
# ------------------------------------------------------------------
test_cols = [c for c in df.columns if c.startswith("test/")]
test_rows = df[test_cols].dropna(how="all") if test_cols else df.iloc[0:0]
print("\n=== KET QUA TREN TEST SET ===")
if len(test_rows) > 0:
    final_test = test_rows.iloc[-1]
    for col in sorted(test_cols):
        val = final_test.get(col)
        if pd.notna(val):
            print(f"  {col:28s}: {val:.4f}")
else:
    print("  Chua co ket qua test trong metrics.csv.")
    print("  -> Kiem tra: train() da chay xong chua? run_test mac dinh = True nen se tu")
    print("     chay 1 lan tren test split ngay sau khi train xong epoch cuoi.")


Cell 9 - Load `best_weights` de du doan thu

In [ ]:
import supervision as sv
from PIL import Image

inference_model = RFDETRSegSmall(pretrain_weights=best_weights_path)  # THAY DOI: Small thay vi Medium

test_images = (
    glob.glob(os.path.join(DATASET_DIR, "test", "*.jpg"))
    + glob.glob(os.path.join(DATASET_DIR, "test", "*.png"))
)
if test_images:
    sample_image_path = test_images[0]
    detections = inference_model.predict(sample_image_path, threshold=0.5)
    image = Image.open(sample_image_path)
    annotated = sv.MaskAnnotator().annotate(image.copy(), detections)
    annotated = sv.LabelAnnotator().annotate(annotated, detections)
    annotated
else:
    print("Khong tim thay anh test de demo - chay predict truc tiep tren anh cua ban nhe.")
